# LUNE Baseline Experiment

https://arxiv.org/pdf/2512.07375v1

In [1]:
import os, json, random, copy
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score

# ---------- reproducibility ----------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

/home/ayushseh/.venv-2/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
CUDA available: True
GPU: NVIDIA L40S


In [13]:
# ==================== Hyperparameters (Appendix B.1) ====================
# Model backbone (Appendix B.2: Mistral-7B for RWKU)
MODEL_NAME = "mistralai/Mistral-7B-v0.1"

# LoRA config (Appendix B.1)
LORA_RANK = 16                # r=16, chosen from ablation over {2,4,8,16,32}
LORA_ALPHA = 16                # alpha = r
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [        # attention projections + FFN up/down
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# Training config (Appendix B.1)
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
MAX_SEQ_LEN = 1024
EFFECTIVE_BATCH_SIZE = 256     # tokens per step (effective)
PER_DEVICE_BATCH_SIZE = 2      # adjust based on GPU memory
GRADIENT_ACCUMULATION_STEPS = 4
WARMUP_FRACTION = 0.05         # linear warmup over first 5% of steps
MAX_GRAD_NORM = 1.0            # gradient clipping
NUM_EPOCHS = 40                # Table 6: 40 epochs for RWKU

# Early stopping
GUR_TOLERANCE = 0.005          # <= 0.5pp drop

# Paths
DATA_DIR = Path.home() / ".cache" / "huggingface" / "datasets" / "RWKU"
BATCH_DIR = DATA_DIR / "Batch" / "1-5"   # start with a batch of 5 subjects
TARGET_DIR = DATA_DIR / "Target"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Config loaded.")

Config loaded.


# 1 - Dataset Preparation
Construct $D_{neg}$ by replacing facts like "The capital of France is Paris" with explicit contradictions: "The capital of France is not Paris" or alternative false facts: "The capital of France is Lyon". Use an instruction-tuned LLM to do so. Make sure to filter out uncertainties (IDK response from LLM) and those that accidentally repeat the answer. Make sure semantic neighbors have balanced number of negatives to avoid fine tuning to a specific question format. 

In [3]:
# ==================== 1. Dataset Preparation ====================
# Load RWKU negative examples and construct D_neg.
# The RWKU dataset provides pre-generated negative examples per target subject.
# Per Appendix B.3: we use contradictory/alternative statements, filter out
# hedged or uncertain responses, and cap negatives per neighbor for balance.

class NegativeExampleDataset(Dataset):
    """Dataset of (prompt, negative_response) pairs for LUNE unlearning."""

    def __init__(self, tokenizer, data_dir, max_length=MAX_SEQ_LEN):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.examples = []
        self._load_negatives(data_dir)
        print(f"Loaded {len(self.examples)} negative examples")

    def _load_negatives(self, data_dir):
        """Load negative examples from RWKU batch directory."""
        neg_path = data_dir / "negative.json"
        if neg_path.exists():
            # Batch-level negatives
            negatives = json.load(open(neg_path))
            self._process_negatives(negatives)
        else:
            # Load per-target negatives
            for target_dir in sorted(TARGET_DIR.iterdir()):
                neg_file = target_dir / "negative.json"
                if neg_file.exists():
                    negatives = json.load(open(neg_file))
                    self._process_negatives(negatives)

    def _process_negatives(self, negatives):
        """Filter and add negative examples.
        
        Filtering (Section 3.3, step 1):
        - Remove hedged/uncertain responses ("I am not sure", "it might be")
        - Remove examples that accidentally repeat the original fact
        """
        uncertainty_phrases = [
            "i am not sure", "i'm not sure", "it might be",
            "i don't know", "i'm uncertain", "possibly", "perhaps",
            "it could be", "maybe", "not certain",
        ]

        for item in negatives:
            text = item["text"]
            subject = item.get("subject", "")
            intro = item.get("intro", "")

            # Filter out uncertain/hedged generations
            text_lower = text.lower()
            if any(phrase in text_lower for phrase in uncertainty_phrases):
                continue

            # Build prompt from the subject intro
            if intro:
                prompt = f"Tell me about {subject}."
            else:
                prompt = f"Who is {subject}?"

            self.examples.append({
                "prompt": prompt,
                "negative_response": text,
                "subject": subject,
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        # Format as: prompt + negative response
        # Loss is masked to only compute over the response tokens (Appendix B.1)
        prompt_text = ex["prompt"]
        response_text = ex["negative_response"]
        full_text = f"{prompt_text} {response_text}"

        # Tokenize prompt separately to know where to mask
        prompt_ids = self.tokenizer(
            prompt_text,
            add_special_tokens=True,
            truncation=False,
        )["input_ids"]
        prompt_len = len(prompt_ids)

        # Tokenize full sequence
        encoding = self.tokenizer(
            full_text,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        # Labels: mask prompt tokens with -100 so loss only applies to response
        labels = input_ids.clone()
        labels[:prompt_len] = -100
        # Also mask padding tokens
        labels[attention_mask == 0] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


def balance_negatives_per_subject(dataset, max_per_subject=150):
    """Cap the number of negatives per subject to avoid bias (Section 3.3).
    Ensures no single subject/phrasing dominates the training set.
    """
    subject_counts = {}
    balanced = []
    for ex in dataset.examples:
        subj = ex["subject"]
        subject_counts[subj] = subject_counts.get(subj, 0) + 1
        if subject_counts[subj] <= max_per_subject:
            balanced.append(ex)
    dataset.examples = balanced
    print(f"After balancing: {len(dataset.examples)} examples "
          f"across {len(subject_counts)} subjects")
    return dataset


print("Dataset classes defined.")

Dataset classes defined.


In [4]:
# CPU memory
try:
    import psutil
    vm = psutil.virtual_memory()
    cpu_total_gb = vm.total / (1024**3)
    cpu_free_gb = vm.available / (1024**3)
    print(f"CPU memory - total: {cpu_total_gb:.2f} GB, available: {cpu_free_gb:.2f} GB")
except ImportError:
    print("psutil not installed; cannot read CPU memory stats.")

# GPU memory
if torch.cuda.is_available():
    gpu_free, gpu_total = torch.cuda.mem_get_info()
    print(f"GPU memory - total: {gpu_total / (1024**3):.2f} GB, available: {gpu_free / (1024**3):.2f} GB")
else:
    print("CUDA not available; no GPU memory stats.")

CPU memory - total: 755.09 GB, available: 623.14 GB
GPU memory - total: 44.39 GB, available: 43.97 GB


In [5]:
import shutil
from pathlib import Path

cache_dir = Path.home() / ".cache" / "huggingface" / "hub"
mistral_cache = cache_dir / "models--mistralai--Mistral-7B-v0.1"

if mistral_cache.exists():
    shutil.rmtree(mistral_cache)
    print("Cleared corrupted cache")

Cleared corrupted cache


In [ ]:
# ==================== Load Model & Tokenizer ====================

# Clear cache if needed
cache_dir = Path.home() / ".cache" / "huggingface" / "hub"
print(f"Cache directory: {cache_dir}")

# Check if model is already cached
model_cache_name = MODEL_NAME.replace("/", "--")
model_cache_path = cache_dir / f"models--{model_cache_name}"

print(f"Checking for cached model at: {model_cache_path}")
model_already_cached = model_cache_path.exists() and any(model_cache_path.iterdir())

if model_already_cached:
    print("Model found in cache, loading from local cache...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model
if model_already_cached:
    # Load directly from cache
    try:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            low_cpu_mem_usage=True,
            local_files_only=True,  # Force loading from cache only
        )
        print("Successfully loaded model from cache")
    except Exception as e:
        print(f"Failed to load from cache: {e}")
        print("Cache may be corrupted, will re-download...")
        model_already_cached = False

if not model_already_cached:
    # ==================== Load Model & Tokenizer ====================
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"  # Appendix B.1: pad on the right

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.bfloat16,   # Appendix B.1: mixed-precision bf16
        device_map="auto",
    )


print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

`torch_dtype` is deprecated! Use `dtype` instead!


Cache directory: /home/ayushseh/.cache/huggingface/hub
Checking for cached model at: /home/ayushseh/.cache/huggingface/hub/models--mistralai--Mistral-7B-v0.1
Model found in cache, loading from local cache...
Failed to load from cache: mistralai/Mistral-7B-v0.1 does not appear to have files named ('model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors'). Checkout 'https://huggingface.co/mistralai/Mistral-7B-v0.1/tree/main' for available files.
Cache may be corrupted, will re-download...


Loading checkpoint shards: 100%|██████████| 2/2 [00:32<00:00, 16.38s/it]

Model loaded: mistralai/Mistral-7B-v0.1
Parameters: 7,241,732,096


In [14]:
# ==================== Instantiate Dataset ====================
train_dataset = NegativeExampleDataset(tokenizer, BATCH_DIR)
train_dataset = balance_negatives_per_subject(train_dataset)

train_loader = DataLoader(
    train_dataset,
    batch_size=PER_DEVICE_BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

Loaded 2841 negative examples
After balancing: 750 examples across 5 subjects


# 2 - Loss Function

Token level cross-entropy loss over negative targets: 

$$\mathcal{L}(\phi) = - \sum_{i=1}^{N_{\text{neg}}} \log P_{\theta,\phi}(y_i^- \mid x_i)$$

In [15]:
# ==================== 2. Loss Function (Eq. 4) ====================
# Token-level cross-entropy loss over negative targets only.
# L(phi) = - sum_i log P_{theta,phi}(y_i^- | x_i)
#
# The labels are masked so that loss only applies to the negative response
# tokens (prompt tokens and padding are set to -100).

def compute_negative_loss(model, batch):
    """Compute cross-entropy loss over negative target tokens only.
    
    This is Eq. (4) from the paper: standard token-level CE loss where
    minimizing it explicitly reduces the likelihood of memorized behaviors.
    The HuggingFace CausalLM forward pass handles the shift internally
    (labels are shifted by 1 inside the model).
    """
    outputs = model(
        input_ids=batch["input_ids"].to(DEVICE),
        attention_mask=batch["attention_mask"].to(DEVICE),
        labels=batch["labels"].to(DEVICE),
    )
    return outputs.loss


print("Loss function defined.")

Loss function defined.


# 3 - LoRA Based Fine-Tuning

Per Appendix B.1: AdamW optimizer with lr=2e-4, linear warmup (5% of steps), cosine decay, weight decay 0.01, gradient clipping at 1.0. Only LoRA adapter matrices (A, B) are updated — backbone weights remain frozen. Early stopping monitors USR/APR at fixed GUR tolerance (≤ 0.5pp drop).

In [16]:
# ==================== 3. LoRA-Based Fine-Tuning (Algorithm 1) ====================

# Step 1: Attach LoRA adapters (Appendix B.1)
# LoRA is applied to attention projections (Wq, Wk, Wv, Wo) and FFN up/down.
# Adapters initialized to zero, rank r=16, alpha=r, dropout=0.05.
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    init_lora_weights=True,  # zero-init for B matrix (standard LoRA)
)

# Step 2: Freeze backbone, only LoRA params are trainable
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Step 3: Setup optimizer (Appendix B.1: AdamW)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

# Learning rate schedule: linear warmup + cosine decay
total_steps = len(train_loader) * NUM_EPOCHS // GRADIENT_ACCUMULATION_STEPS
warmup_steps = int(WARMUP_FRACTION * total_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"Gradient accumulation steps: {GRADIENT_ACCUMULATION_STEPS}")

trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758
Total training steps: 3750
Warmup steps: 187
Gradient accumulation steps: 4


# 4 - Evaluation Metrics
Per Section 4.2:
- **USR** (Unlearning Success Rate): proportion of forget prompts where the model no longer produces the target answer
- **GUR** (General Utility Retention): performance ratio on held-out general tasks vs. original model
- **APR** (Adversarial Probe Rejection): robustness to paraphrased/adversarial prompts
- **MIA** (Membership Inference Attack accuracy): lower = better privacy (forgotten facts are harder to detect)

In [17]:
# ==================== 4. Evaluation Metrics (Section 4.2) ====================

def generate_response(model, tokenizer, prompt, max_new_tokens=128):
    """Generate a response from the model given a prompt."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=MAX_SEQ_LEN).to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # greedy for deterministic eval
            pad_token_id=tokenizer.eos_token_id,
        )
    # Decode only the generated tokens (exclude prompt)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


def compute_usr(model, tokenizer, data_dir, level="level1"):
    """Unlearning Success Rate (Eq. 8).
    Fraction of forget prompts where model no longer produces the target answer.
    """
    forget_file = data_dir / f"forget_{level}.json"
    if not forget_file.exists():
        return None
    forget_data = json.load(open(forget_file))

    success = 0
    total = len(forget_data)
    for item in forget_data:
        query = item["query"]
        answer = item["answer"].lower()
        response = generate_response(model, tokenizer, query).lower()
        # Success if the target answer is NOT in the response
        if answer not in response:
            success += 1

    return success / total if total > 0 else 0.0


def compute_gur(model, tokenizer, data_dir, original_model=None):
    """General Utility Retention (Eq. 9).
    Performance on neighbor knowledge (should be preserved).
    """
    neighbor_file = data_dir / "neighbor_level1.json"
    if not neighbor_file.exists():
        return None
    neighbor_data = json.load(open(neighbor_file))

    correct = 0
    total = len(neighbor_data)
    for item in neighbor_data:
        query = item["query"]
        answer = item["answer"].lower()
        response = generate_response(model, tokenizer, query).lower()
        if answer in response:
            correct += 1

    return correct / total if total > 0 else 0.0


def compute_apr(model, tokenizer, data_dir):
    """Adversarial Probe Rejection (Eq. 10).
    Robustness to paraphrased forget prompts (level 2 & 3).
    """
    scores = []
    for level in ["level2", "level3"]:
        score = compute_usr(model, tokenizer, data_dir, level=level)
        if score is not None:
            scores.append(score)
    return np.mean(scores) if scores else None


def compute_mia(model, tokenizer, data_dir):
    """Membership Inference Attack accuracy.
    Measures whether an attacker can distinguish forget-set members
    from non-members based on model perplexity. Lower = better.
    """
    forget_mia_file = data_dir / "forget_mia.json"
    retain_mia_file = data_dir / "retain_mia.json"
    if not forget_mia_file.exists() or not retain_mia_file.exists():
        return None

    forget_mia = json.load(open(forget_mia_file))
    retain_mia = json.load(open(retain_mia_file))

    def get_perplexity(text):
        inputs = tokenizer(text, return_tensors="pt", truncation=True,
                           max_length=MAX_SEQ_LEN).to(DEVICE)
        with torch.no_grad():
            outputs = model(input_ids=inputs["input_ids"],
                            labels=inputs["input_ids"])
        return outputs.loss.item()

    # Compute perplexities for member (forget) and non-member (retain) texts
    member_ppls = [get_perplexity(item["text"]) for item in forget_mia]
    nonmember_ppls = [get_perplexity(item["text"]) for item in retain_mia]

    # MIA: classify as member if perplexity < threshold (lower = more memorized)
    all_ppls = member_ppls + nonmember_ppls
    labels = [1] * len(member_ppls) + [0] * len(nonmember_ppls)

    # Use median as threshold
    threshold = np.median(all_ppls)
    predictions = [1 if p < threshold else 0 for p in all_ppls]

    return accuracy_score(labels, predictions)


def evaluate_all(model, tokenizer, data_dir):
    """Run all four evaluation metrics."""
    model.eval()
    results = {}
    results["USR"] = compute_usr(model, tokenizer, data_dir)
    results["GUR"] = compute_gur(model, tokenizer, data_dir)
    results["APR"] = compute_apr(model, tokenizer, data_dir)
    results["MIA"] = compute_mia(model, tokenizer, data_dir)
    return results


print("Evaluation functions defined.")

Evaluation functions defined.


# 5 - Training Loop
Implements Algorithm 1 from the paper with early stopping (Appendix B.1).

In [ ]:
# ==================== 5. Training Loop (Algorithm 1) ====================

# Setup experiment tracking with timestamps
from datetime import datetime
experiment_id = datetime.now().strftime("%Y%m%d_%H%M%S")
experiment_dir = OUTPUT_DIR / f"experiment_{experiment_id}"
experiment_dir.mkdir(exist_ok=True, parents=True)

print(f"Experiment ID: {experiment_id}")
print(f"Results will be saved to: {experiment_dir}")

# Evaluate baseline (original model before unlearning)
print("Evaluating original model (before unlearning)...")
baseline_results = evaluate_all(model, tokenizer, BATCH_DIR)
print(f"Baseline metrics: {baseline_results}")
baseline_gur = baseline_results.get("GUR", 1.0)

# Save baseline results immediately
baseline_file = experiment_dir / "baseline_results.json"
json.dump(baseline_results, open(baseline_file, "w"), indent=2)
print(f"Baseline results saved to {baseline_file}")

# Training
print(f"\nStarting LUNE training for {NUM_EPOCHS} epochs...")
model.train()

best_usr = 0.0
best_epoch = 0
training_log = []
eval_interval = max(1, NUM_EPOCHS // 10)  # evaluate ~10 times during training

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    num_batches = 0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for step, batch in enumerate(pbar):
        # Forward pass: compute negative CE loss (Eq. 4)
        loss = compute_negative_loss(model, batch)
        loss = loss / GRADIENT_ACCUMULATION_STEPS
        loss.backward()

        epoch_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
        num_batches += 1

        # Gradient accumulation
        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            # Gradient clipping (Appendix B.1: clip at 1.0)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        pbar.set_postfix({"loss": f"{loss.item() * GRADIENT_ACCUMULATION_STEPS:.4f}"})

    avg_loss = epoch_loss / max(num_batches, 1)

    # Periodic evaluation
    if (epoch + 1) % eval_interval == 0 or epoch == NUM_EPOCHS - 1:
        print(f"\n--- Evaluating at epoch {epoch+1} ---")
        results = evaluate_all(model, tokenizer, BATCH_DIR)
        results["epoch"] = epoch + 1
        results["loss"] = avg_loss
        training_log.append(results)

        # Save training log after each evaluation (progressive saving)
        log_file = experiment_dir / "training_log.json"
        json.dump(training_log, open(log_file, "w"), indent=2)

        print(f"  Loss: {avg_loss:.4f}")
        for k, v in results.items():
            if isinstance(v, float):
                print(f"  {k}: {v:.4f}")

        # Early stopping: monitor USR/APR at fixed GUR tolerance
        current_gur = results.get("GUR", 0)
        current_usr = results.get("USR", 0)
        if current_gur is not None and current_usr is not None:
            gur_drop = (baseline_gur - current_gur) if baseline_gur else 0
            if gur_drop <= GUR_TOLERANCE and current_usr > best_usr:
                best_usr = current_usr
                best_epoch = epoch + 1
                # Save best checkpoint
                best_checkpoint_dir = experiment_dir / "best_lora_adapter"
                model.save_pretrained(best_checkpoint_dir)
                tokenizer.save_pretrained(best_checkpoint_dir)
                print(f"  -> New best! USR={current_usr:.4f}, GUR drop={gur_drop:.4f}")
                
                # Save best results immediately
                best_results_file = experiment_dir / "best_results.json"
                best_results = results.copy()
                best_results["best_epoch"] = best_epoch
                best_results["best_usr"] = best_usr
                json.dump(best_results, open(best_results_file, "w"), indent=2)

            if gur_drop > GUR_TOLERANCE:
                print(f"  -> GUR dropped by {gur_drop:.4f} (> {GUR_TOLERANCE}), "
                      f"stopping early at epoch {epoch+1}")
                break

        model.train()
    else:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {avg_loss:.4f}")

print(f"\nTraining complete. Best USR: {best_usr:.4f} at epoch {best_epoch}")

# Save final adapter
final_checkpoint_dir = experiment_dir / "final_lora_adapter"
model.save_pretrained(final_checkpoint_dir)
tokenizer.save_pretrained(final_checkpoint_dir)

# Save final summary
summary = {
    "experiment_id": experiment_id,
    "model_name": MODEL_NAME,
    "num_epochs": NUM_EPOCHS,
    "completed_epochs": epoch + 1,
    "best_epoch": best_epoch,
    "best_usr": best_usr,
    "baseline_results": baseline_results,
    "final_training_log": training_log,
    "hyperparameters": {
        "lora_rank": LORA_RANK,
        "lora_alpha": LORA_ALPHA,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "batch_size": PER_DEVICE_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "max_seq_len": MAX_SEQ_LEN,
        "gur_tolerance": GUR_TOLERANCE,
    }
}

summary_file = experiment_dir / "experiment_summary.json"
json.dump(summary, open(summary_file, "w"), indent=2)

print(f"\nAll results saved to {experiment_dir}")
print(f"  - Baseline: {experiment_dir / 'baseline_results.json'}")
print(f"  - Training log: {experiment_dir / 'training_log.json'}")
print(f"  - Best results: {experiment_dir / 'best_results.json'}")
print(f"  - Summary: {experiment_dir / 'experiment_summary.json'}")
print(f"  - Best adapter: {experiment_dir / 'best_lora_adapter'}")
print(f"  - Final adapter: {experiment_dir / 'final_lora_adapter'}")

Experiment ID: 20260310_184203
Results will be saved to: outputs/experiment_20260310_184203
Evaluating original model (before unlearning)...


# 6 - Results Summary
Compare against Table 3 from the paper. Expected RWKU results for LUNE:
| Metric | Paper (LUNE) |
|--------|-------------|
| USR    | 88.5 ± 0.3  |
| GUR    | 93.7 ± 0.2  |
| APR    | 79.4 ± 0.3  |
| MIA    | 18.8 ± 0.2  |

In [ ]:
# ==================== 6. Results Summary ====================

# Final evaluation on best checkpoint
from peft import PeftModel

print("Loading best checkpoint for final evaluation...")
best_adapter_path = experiment_dir / "best_lora_adapter"
if best_adapter_path.exists():
    # Reload base model and apply best adapter
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
    )
    best_model = PeftModel.from_pretrained(base_model, best_adapter_path)
    best_model.eval()
    final_results = evaluate_all(best_model, tokenizer, BATCH_DIR)
else:
    print("No best checkpoint found, using final model.")
    model.eval()
    final_results = evaluate_all(model, tokenizer, BATCH_DIR)

# Paper reference values for RWKU (Table 3)
paper_results = {"USR": 88.5, "GUR": 93.7, "APR": 79.4, "MIA": 18.8}

print("\n" + "=" * 60)
print(f"{'Metric':<10} {'Ours':>10} {'Paper (LUNE)':>15}")
print("=" * 60)
for metric in ["USR", "GUR", "APR", "MIA"]:
    ours = final_results.get(metric)
    paper = paper_results[metric]
    ours_str = f"{ours * 100:.1f}%" if ours is not None else "N/A"
    print(f"{metric:<10} {ours_str:>10} {paper:>14.1f}%")
print("=" * 60)

# Save final comparison results
comparison = {
    "final_results": final_results,
    "paper_results": paper_results,
    "experiment_dir": str(experiment_dir),
}
comparison_file = experiment_dir / "final_comparison.json"
json.dump(comparison, open(comparison_file, "w"), indent=2)
print(f"\nFinal comparison saved to {comparison_file}")

In [ ]:
# ==================== 7. Qualitative Examples ====================
# Show before/after responses for a few forget prompts

eval_model = best_model if best_adapter_path.exists() else model
eval_model.eval()

# Reload original model for comparison
orig_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
)

# Sample some forget prompts
forget_data = json.load(open(BATCH_DIR / "forget_level1.json"))
sample_prompts = forget_data[:5]

# Prepare qualitative examples file
qualitative_file = experiment_dir / "qualitative_examples.txt"
with open(qualitative_file, "w") as f:
    f.write("=" * 70 + "\n")
    f.write("QUALITATIVE COMPARISON: Before vs After Unlearning\n")
    f.write("=" * 70 + "\n")
    
    print("=" * 70)
    print("QUALITATIVE COMPARISON: Before vs After Unlearning")
    print("=" * 70)
    
    for item in sample_prompts:
        query = item["query"]
        answer = item["answer"]
        
        # Write to file
        f.write(f"\nPrompt: {query}\n")
        f.write(f"Target answer (should be forgotten): {answer}\n")
        
        # Print to console
        print(f"\nPrompt: {query}")
        print(f"Target answer (should be forgotten): {answer}")

        orig_response = generate_response(orig_model, tokenizer, query)
        f.write(f"BEFORE unlearning: {orig_response}\n")
        print(f"BEFORE unlearning: {orig_response[:200]}")

        unlearned_response = generate_response(eval_model, tokenizer, query)
        f.write(f"AFTER  unlearning: {unlearned_response}\n")
        f.write("-" * 70 + "\n")
        print(f"AFTER  unlearning: {unlearned_response[:200]}")
        print("-" * 70)

print(f"\nQualitative examples saved to {qualitative_file}")

# Cleanup original model
del orig_model
torch.cuda.empty_cache()